# LeetCode 1823 — Find the Winner of the Circular Game bằng Python

Link đề: https://leetcode.com/problems/find-the-winner-of-the-circular-game/ · Độ khó: **Medium** · Chủ đề: Array, Math, Recursion, Queue, Simulation

Skeleton LeetCode:

```python
class Solution:
    def findTheWinner(self, n: int, k: int) -> int:
        ...
```

Đây là **bài toán Josephus** kinh điển. Lộ trình: **mô phỏng đếm từng người** trên vòng tròn có đánh dấu → **list + `pop(index)`** `O(n²)` → **`deque` xoay vòng** `O(n·k)` → **công thức truy hồi Josephus** (đệ quy) → **vòng lặp `O(n)` thời gian, `O(1)` bộ nhớ** (trả lời follow-up).

## 1. Mục tiêu học

1. Mô phỏng chính xác trò chơi trên vòng tròn bằng **chỉ số modulo** (`(idx + k − 1) % size`).
2. Dùng `collections.deque.rotate` để mô phỏng "đếm vòng" gọn gàng.
3. Suy ra **công thức truy hồi Josephus** `J(n) = (J(n−1) + k) mod n` bằng lập luận **đánh số lại** sau mỗi lượt loại.
4. Chuyển đệ quy thành **vòng lặp** để đạt `O(1)` bộ nhớ và tránh giới hạn đệ quy.
5. Phân biệt rõ chỉ số **0-based** (trong công thức) và **1-based** (đề bài).

## 2. Đề bài & ví dụ

Có `n` người bạn ngồi thành **vòng tròn**, đánh số `1..n` theo chiều kim đồng hồ (đi tiếp từ người `n` là về người `1`).

Luật chơi:
1. Bắt đầu ở người số 1.
2. Đếm `k` người theo chiều kim đồng hồ, **tính cả người bắt đầu**. Việc đếm có thể quay vòng và đếm một người nhiều lần.
3. Người cuối cùng được đếm **rời vòng** và thua.
4. Nếu còn nhiều hơn một người, quay lại bước 2, **bắt đầu từ người ngay sau** (theo chiều kim đồng hồ) người vừa rời.
5. Ngược lại, người cuối cùng còn lại thắng.

Cho `n` và `k`, trả về **người thắng**.

**Ví dụ 1:** `n = 5, k = 2` → `3` (lần lượt rời: 2, 4, 1, 5).

**Ví dụ 2:** `n = 6, k = 5` → `1` (lần lượt rời: 5, 4, 6, 2, 3).

**Ràng buộc:** `1 ≤ k ≤ n ≤ 500`. **Follow-up:** giải trong thời gian tuyến tính, bộ nhớ hằng số?

In [1]:
import random
import sys
import time
from collections import deque
from typing import List

examples = [((5, 2), 3), ((6, 5), 1)]
for (n, k), want in examples:
    print(f"n={n}, k={k} -> kỳ vọng {want}")

n=5, k=2 -> kỳ vọng 3
n=6, k=5 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Chỉ số vòng tròn bằng modulo

Trên vòng `size` phần tử (chỉ số `0..size−1`), đi `t` bước từ `idx` tới `(idx + t) % size`. "Đếm `k` người tính cả người bắt đầu" = đi `k − 1` bước.

In [2]:
size, idx, k = 5, 3, 4
print("đếm", k, "người từ chỉ số", idx, "-> dừng ở", (idx + k - 1) % size)

đếm 4 người từ chỉ số 3 -> dừng ở 1


### 3.2 `list.pop(i)` và vị trí sau khi xóa

Xóa phần tử ở chỉ số `i` làm các phần tử phía sau **dồn lên**: phần tử "ngay sau" người bị xóa giờ nằm ở **chính chỉ số `i`** (hoặc `0` nếu `i` là cuối list). `pop(i)` tốn `O(size − i)`.

In [3]:
people = [1, 2, 3, 4, 5]
i = 1
removed = people.pop(i)
print("xóa", removed, "->", people, "| người ngay sau bây giờ ở chỉ số", i % len(people), "=", people[i % len(people)])

xóa 2 -> [1, 3, 4, 5] | người ngay sau bây giờ ở chỉ số 1 = 3


### 3.3 `deque.rotate`

`dq.rotate(-t)` xoay **trái** `t` bước: `t` phần tử đầu chuyển xuống cuối. Đưa người thứ `k` lên đầu bằng `rotate(-(k−1))` rồi `popleft()`.

In [4]:
dq = deque([1, 2, 3, 4, 5])
dq.rotate(-(2 - 1))
print(dq, "-> popleft:", dq.popleft(), "->", dq)

deque([3, 4, 5, 1]) -> popleft: 2 -> deque([3, 4, 5, 1])


### 3.4 Đánh số lại (relabel) và dịch modulo

Nếu trên vòng mới, người ở vị trí `x` (0-based, tính từ người bắt đầu mới) ứng với vị trí `(x + s) % n` trên vòng cũ, thì ta "dịch" kết quả bằng `+ s` rồi `% n`. Đây là chìa khóa của công thức Josephus.

In [5]:
n, s = 5, 2
for x in range(n - 1):
    print(f"vị trí mới {x} -> vị trí cũ {(x + s) % n}")

vị trí mới 0 -> vị trí cũ 2
vị trí mới 1 -> vị trí cũ 3
vị trí mới 2 -> vị trí cũ 4
vị trí mới 3 -> vị trí cũ 0


### 3.5 Giới hạn đệ quy

Công thức đệ quy sâu `n` tầng; `n ≤ 500` thì an toàn (giới hạn 1000), nhưng với follow-up `n` lớn phải dùng vòng lặp.

In [6]:
print("recursion limit:", sys.getrecursionlimit())

recursion limit: 1000


### 3.6 Đếm quay vòng khi `k` lớn hơn số người còn lại

Ở cuối trò chơi, vòng có thể chỉ còn 2–3 người trong khi `k` lớn (Ví dụ 2: `k = 5`, còn 2 người). "Đếm `k` người" khi đó đi vòng nhiều lần; modulo xử lý gọn: `(idx + k − 1) % size`.

In [7]:
people, idx, k = [3, 1], 0, 5
for step in range(k):
    print(f"đếm {step + 1}: người {people[(idx + step) % len(people)]}")
print("người rời:", people[(idx + k - 1) % len(people)])

đếm 1: người 3
đếm 2: người 1
đếm 3: người 3
đếm 4: người 1
đếm 5: người 3
người rời: 3


## 4. Ý tưởng nền tảng / trực giác

Mô phỏng tay Ví dụ 1 (`n = 5, k = 2`), vòng ghi dạng list, `→` là người bắt đầu đếm:

| lượt | vòng | bắt đầu | đếm 2 người | rời |
|---|---|---|---|---|
| 1 | `[1,2,3,4,5]` | 1 | 1, 2 | **2** |
| 2 | `[1,3,4,5]` | 3 | 3, 4 | **4** |
| 3 | `[1,3,5]` | 5 | 5, 1 | **1** |
| 4 | `[3,5]` | 3 | 3, 5 | **5** |

Còn lại 3 ⇒ thắng. ✔

**Quan sát dẫn tới công thức:** sau lượt đầu, vòng còn `n − 1` người và trò chơi **giống hệt** trò chơi với `n − 1` người — chỉ khác là bắt đầu ở người ngay sau người vừa rời. Nếu biết người thắng của trò chơi `n − 1` người (theo vị trí tính từ người bắt đầu), ta "dịch" về vị trí trên vòng `n` người.

## 5. Cách 1 — Brute Force / baseline: đếm từng người trên vòng có đánh dấu

**Ý tưởng:** mảng `alive[1..n]`; con trỏ `cur` đi từng bước quanh vòng, **bỏ qua** người đã rời, đếm tới `k` thì loại. Làm đúng nghĩa đen của luật chơi.

In [8]:
def winner_mark(n, k, verbose=False):
    alive = [True] * (n + 1)          # bỏ chỉ số 0
    cur = 1                           # người bắt đầu đếm
    for _ in range(n - 1):
        count = 0
        while True:
            if alive[cur]:
                count += 1
                if count == k:
                    break
            cur = cur % n + 1         # đi tới người kế tiếp (vòng 1..n)
        alive[cur] = False
        if verbose:
            print("rời:", cur)
        # người bắt đầu lượt sau: người còn sống ngay sau cur
        cur = cur % n + 1
        while not alive[cur]:
            cur = cur % n + 1
    return alive.index(True, 1)

print("=>", winner_mark(5, 2, verbose=True))
print("=>", winner_mark(6, 5))

rời: 2
rời: 4
rời: 1
rời: 5
=> 3
=> 1


**Độ phức tạp:** mỗi lượt đếm `k` người còn sống nhưng có thể phải **đi qua** nhiều người đã rời — tới `O(n)` bước mỗi lần đếm một người ⇒ worst `O(n · k · n)`… cụ thể mỗi lượt `O(n + k)` bước khi đếm quay vòng nhiều, tổng cỡ `O(n² + n·k)`. Với `n = 500` vẫn nhanh, nhưng ta lãng phí ở chỗ duyệt qua người đã rời.

**Nút thắt:** người đã rời vẫn chiếm chỗ ⇒ **xóa hẳn** khỏi cấu trúc dữ liệu.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (mô phỏng gọn):** dùng list những người **còn lại**; người bị loại ở chỉ số `idx = (idx + k − 1) % size`; sau khi `pop(idx)`, người bắt đầu lượt sau đã nằm sẵn ở chỉ số `idx % size_mới`.

**Insight 2 (truy hồi Josephus, 0-based):** gọi `J(m)` là **vị trí (0-based, tính từ người bắt đầu)** của người thắng trong trò chơi `m` người. Khi đó:
- `J(1) = 0`;
- `J(m) = (J(m − 1) + k) mod m`.

*Lý do:* lượt đầu trong trò chơi `m` người loại người ở vị trí `(k − 1) mod m`; trò chơi còn lại có `m − 1` người, bắt đầu ở vị trí `k mod m`. Người ở vị trí `x` của trò chơi mới chính là vị trí `(x + k) mod m` của trò chơi cũ (vì ta bắt đầu đếm lại từ `k mod m` và bỏ qua người đã rời — người đã rời nằm ngay trước điểm bắt đầu mới nên phép dịch không "vấp" phải họ).

**Quy tắc:** đáp án đề bài (1-based) là `J(n) + 1`.

**Invariant cho vòng lặp:** sau khi xử lý `m`, biến `res` = `J(m)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — List + `pop(index)` (`O(n²)`)

In [9]:
def winner_list(n, k, verbose=False):
    people = list(range(1, n + 1))
    idx = 0
    while len(people) > 1:
        idx = (idx + k - 1) % len(people)
        out = people.pop(idx)
        if verbose:
            print(f"rời {out}, còn {people}, bắt đầu tiếp ở chỉ số {idx % len(people)}")
        idx %= len(people)           # người ngay sau đã dồn vào chỉ số idx
    return people[0]

print("=>", winner_list(6, 5, verbose=True))

rời 5, còn [1, 2, 3, 4, 6], bắt đầu tiếp ở chỉ số 4
rời 4, còn [1, 2, 3, 6], bắt đầu tiếp ở chỉ số 3
rời 6, còn [1, 2, 3], bắt đầu tiếp ở chỉ số 0
rời 2, còn [1, 3], bắt đầu tiếp ở chỉ số 1
rời 3, còn [1], bắt đầu tiếp ở chỉ số 0
=> 1


**Độ phức tạp:** `n − 1` lần `pop`, mỗi lần `O(n)` ⇒ `O(n²)`; bộ nhớ `O(n)`.

### 7.2 Cách 3 — `deque` xoay vòng (`O(n·k)`)

In [10]:
class SolutionDeque:
    def findTheWinner(self, n: int, k: int) -> int:
        dq = deque(range(1, n + 1))
        while len(dq) > 1:
            dq.rotate(-(k - 1))          # đưa người thứ k lên đầu
            dq.popleft()                 # người đó rời; người kế tiếp thành đầu deque
        return dq[0]

for (n, k), want in examples:
    print(SolutionDeque().findTheWinner(n, k), want)

3 3
1 1


**Độ phức tạp:** `rotate(t)` tốn `O(min(t, size − t))`; tổng `O(n · k)`. Code rất "đúng nghĩa đen", dễ kiểm chứng.

### 7.3 Cách 4 — Truy hồi Josephus (đệ quy, `O(n)` thời gian, `O(n)` stack)

In [11]:
class SolutionRecursive:
    def findTheWinner(self, n: int, k: int) -> int:
        def J(m):                     # vị trí 0-based của người thắng trong trò chơi m người
            if m == 1:
                return 0
            return (J(m - 1) + k) % m
        return J(n) + 1

for (n, k), want in examples:
    print(SolutionRecursive().findTheWinner(n, k), want)

3 3
1 1


### 7.4 Cách 5 — Vòng lặp (`O(n)` thời gian, `O(1)` bộ nhớ) — trả lời follow-up

In [12]:
class SolutionIterative:
    def findTheWinner(self, n: int, k: int) -> int:
        res = 0                       # J(1) = 0
        for m in range(2, n + 1):
            res = (res + k) % m       # J(m) từ J(m-1)
        return res + 1

for (n, k), want in examples:
    print(SolutionIterative().findTheWinner(n, k), want)

3 3
1 1


Đây chính là "lời giải tuyến tính, bộ nhớ hằng" mà follow-up hỏi.

### 7.5 Cách 6 — Cây Fenwick: thứ tự bị loại trong `O(n log n)`

Công thức Josephus chỉ cho **người thắng**. Nếu cần cả **thứ tự bị loại** với `n` lớn, `list.pop` tốn `O(n²)`. Ý tưởng: giữ cây Fenwick (BIT) đánh dấu 1 cho người còn sống; "người còn sống thứ `t`" tìm được bằng **tìm kiếm nhị phân trên cây** trong `O(log n)`.

- Người bị loại ở lượt hiện tại là người còn sống **thứ** `idx + 1` (với `idx = (idx + k − 1) mod size`, 0-based trong danh sách còn sống).
- Sau khi loại, gán vị trí đó về 0; `size` giảm 1; `idx` giữ nguyên (người kế dồn vào thứ hạng đó) rồi `idx %= size`.

In [13]:
class Fenwick:
    def __init__(self, n):
        self.n = n
        self.t = [0] * (n + 1)
        for i in range(1, n + 1):          # dựng O(n): mọi vị trí = 1
            self.t[i] += 1
            j = i + (i & -i)
            if j <= n:
                self.t[j] += self.t[i]
        self.LOG = 1 << n.bit_length()

    def add(self, i, delta):
        while i <= self.n:
            self.t[i] += delta
            i += i & -i

    def kth(self, k):                       # vị trí nhỏ nhất có tổng tiền tố = k
        pos, step = 0, self.LOG
        while step:
            nxt = pos + step
            if nxt <= self.n and self.t[nxt] < k:
                pos = nxt
                k -= self.t[nxt]
            step >>= 1
        return pos + 1

def elimination_fenwick(n, k):
    bit = Fenwick(n)
    size, idx, order = n, 0, []
    while size > 1:
        idx = (idx + k - 1) % size
        person = bit.kth(idx + 1)
        bit.add(person, -1)
        order.append(person)
        size -= 1
        idx %= size
    return order, bit.kth(1)

print(elimination_fenwick(5, 2))
print(elimination_fenwick(6, 5))

([2, 4, 1, 5], 3)
([5, 4, 6, 2, 3], 1)


**Độ phức tạp:** `n − 1` lượt, mỗi lượt `kth` + `add` `O(log n)` ⇒ `O(n log n)` thời gian, `O(n)` bộ nhớ. Với `n = 10⁵` chạy trong cỡ trăm mili-giây, trong khi list + `pop` mất `O(n²)` ≈ 5·10⁹ thao tác dịch chuyển (dù `pop` chạy bằng C nên vẫn khá nhanh ở cỡ này). Đo thử:

In [14]:
t = time.perf_counter(); order_f, win_f = elimination_fenwick(10**5, 7); t1 = time.perf_counter() - t
assert win_f == SolutionIterative().findTheWinner(10**5, 7)
print(f"Fenwick n=10^5: {1000*t1:.0f} ms, người thắng {win_f}, 5 người rời đầu tiên {order_f[:5]}")

Fenwick n=10^5: 219 ms, người thắng 27152, 5 người rời đầu tiên [7, 14, 21, 28, 35]


## 8. TRACE từng bước

Trace Cách 5 trên Ví dụ 2 (`n = 6, k = 5`):

| m | `J(m−1)` | `(J(m−1) + 5) mod m` | `J(m)` |
|---|---|---|---|
| 1 | — | — | 0 |
| 2 | 0 | 5 mod 2 | 1 |
| 3 | 1 | 6 mod 3 | 0 |
| 4 | 0 | 5 mod 4 | 1 |
| 5 | 1 | 6 mod 5 | 1 |
| 6 | 1 | 6 mod 6 | 0 |

`J(6) = 0` ⇒ người thắng `0 + 1 = 1`. ✔

In [15]:
def trace_josephus(n, k):
    res = 0
    print(f"m=1: J=0")
    for m in range(2, n + 1):
        new = (res + k) % m
        print(f"m={m}: ({res} + {k}) % {m} = {new}")
        res = new
    print("người thắng (1-based):", res + 1)
    return res + 1

trace_josephus(6, 5)

m=1: J=0
m=2: (0 + 5) % 2 = 1
m=3: (1 + 5) % 3 = 0
m=4: (0 + 5) % 4 = 1
m=5: (1 + 5) % 5 = 1
m=6: (1 + 5) % 6 = 0
người thắng (1-based): 1


1

Trace Cách 2 cho Ví dụ 1 để đối chiếu thứ tự rời vòng với đề (2, 4, 1, 5):

In [16]:
winner_list(5, 2, verbose=True)

rời 2, còn [1, 3, 4, 5], bắt đầu tiếp ở chỉ số 1
rời 4, còn [1, 3, 5], bắt đầu tiếp ở chỉ số 2
rời 1, còn [3, 5], bắt đầu tiếp ở chỉ số 0
rời 5, còn [3], bắt đầu tiếp ở chỉ số 0


3

Trace Cách 3 (`deque`) trên Ví dụ 2 để thấy phép xoay:

| lượt | deque trước | `rotate(-4)` | `popleft` (rời) |
|---|---|---|---|
| 1 | `1 2 3 4 5 6` | `5 6 1 2 3 4` | 5 |
| 2 | `6 1 2 3 4` | `4 6 1 2 3` | 4 |
| 3 | `6 1 2 3` | `6 1 2 3` (xoay 4 ≡ 0 mod 4) | 6 |
| 4 | `1 2 3` | `2 3 1` | 2 |
| 5 | `3 1` | `3 1` (xoay 4 ≡ 0 mod 2) | 3 |

Còn `1` ⇒ thắng. ✔

In [17]:
def trace_deque(n, k):
    dq = deque(range(1, n + 1))
    turn = 0
    while len(dq) > 1:
        turn += 1
        before = list(dq)
        dq.rotate(-(k - 1))
        rotated = list(dq)
        out = dq.popleft()
        print(f"lượt {turn}: {before} -> xoay {rotated} -> rời {out}")
    print("thắng:", dq[0])
    return dq[0]

trace_deque(6, 5)

lượt 1: [1, 2, 3, 4, 5, 6] -> xoay [5, 6, 1, 2, 3, 4] -> rời 5
lượt 2: [6, 1, 2, 3, 4] -> xoay [4, 6, 1, 2, 3] -> rời 4
lượt 3: [6, 1, 2, 3] -> xoay [6, 1, 2, 3] -> rời 6
lượt 4: [1, 2, 3] -> xoay [2, 3, 1] -> rời 2
lượt 5: [3, 1] -> xoay [3, 1] -> rời 3
thắng: 1


1

## 9. Code nộp LeetCode

In [18]:
class Solution:
    def findTheWinner(self, n: int, k: int) -> int:
        res = 0                        # J(1) = 0 (0-based)
        for m in range(2, n + 1):
            res = (res + k) % m        # J(m) = (J(m-1) + k) mod m
        return res + 1                 # đổi sang đánh số 1..n

for (n, k), want in examples:
    assert Solution().findTheWinner(n, k) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Đánh số người trong trò chơi `m` người bằng vị trí **0-based tính từ người bắt đầu đếm**: `0, 1, …, m − 1`.

**Lượt đầu:** đếm `k` người tính cả người bắt đầu ⇒ người ở vị trí `r = (k − 1) mod m` rời vòng. Lượt sau bắt đầu ở người ngay sau, tức vị trí cũ `(r + 1) mod m = k mod m`.

**Đánh số lại:** trò chơi còn lại là trò chơi `m − 1` người. Người có vị trí mới `x` (0-based tính từ người bắt đầu mới) là người có vị trí cũ `(x + k) mod m`: đi `x` bước từ `k mod m` trên vòng cũ, và vì người duy nhất đã rời (vị trí `r = k − 1 mod m`) nằm **ngay trước** điểm bắt đầu, `x ≤ m − 2` bước không bao giờ đi tới `r`.

**Quy nạp:** `J(1) = 0` (chỉ còn một người, chính là người bắt đầu). Nếu người thắng của trò chơi `m − 1` người ở vị trí mới `J(m−1)` thì trên vòng `m` người là vị trí `(J(m−1) + k) mod m` ⇒ `J(m) = (J(m−1) + k) mod m`.

Trò chơi gốc bắt đầu ở người số 1 = vị trí 0 ⇒ người thắng là `J(n) + 1`. Các cách mô phỏng (1–3) đúng vì làm đúng luật; mục 12 đối chiếu tất cả.

Kiểm chứng **bước đánh số lại** bằng máy: với mỗi `(m, k)`, mô phỏng một lượt trên vòng `m` người (vị trí `0..m−1`), rồi so danh sách người còn lại (theo thứ tự từ người bắt đầu mới) với ánh xạ `x ↦ (x + k) mod m`.

In [19]:
for m in range(2, 40):
    for k in range(1, 60):
        circle = list(range(m))                 # vị trí cũ, bắt đầu ở 0
        r = (k - 1) % m                         # người rời
        rest = circle[r + 1:] + circle[:r]      # vòng mới, bắt đầu từ người ngay sau
        assert rest == [(x + k) % m for x in range(m - 1)], (m, k)
print("ánh xạ vị trí mới -> cũ là x ↦ (x + k) mod m: đúng với mọi m < 40, k < 60 (kể cả k > m)")

ánh xạ vị trí mới -> cũ là x ↦ (x + k) mod m: đúng với mọi m < 40, k < 60 (kể cả k > m)


Chú ý kiểm thử trên cả `k > m`: khi vòng đã nhỏ hơn `k`, việc đếm quay vòng nhiều lần — công thức vẫn đúng vì mọi phép tính đều lấy modulo `m`.

## 11. Phân tích Time / Space

| Cách | Time | Space |
|---|---|---|
| 1. Đánh dấu, đếm từng người | `O(n² + n·k)` | `O(n)` |
| 2. List + pop | `O(n²)` | `O(n)` |
| 3. Deque rotate | `O(n·k)` | `O(n)` |
| 4. Đệ quy Josephus | `O(n)` | `O(n)` stack |
| 5. Vòng lặp Josephus | `O(n)` | `O(1)` |
| 6. Fenwick (thứ tự loại) | `O(n log n)` | `O(n)` |

In [20]:
for name, f in [("Cách 1", winner_mark), ("Cách 2", winner_list),
                ("Cách 3", SolutionDeque().findTheWinner), ("Cách 4", SolutionRecursive().findTheWinner),
                ("Cách 5", Solution().findTheWinner)]:
    t = time.perf_counter()
    for k in (1, 2, 250, 500):
        f(500, k)
    print(f"{name}: 4 lần n=500 trong {1000*(time.perf_counter()-t):.1f} ms")
t = time.perf_counter(); r = Solution().findTheWinner(10**6, 12345); dt = time.perf_counter() - t
print(f"Cách 5 với n=10^6 (ngoài ràng buộc, minh họa follow-up): {r} trong {1000*dt:.0f} ms")

Cách 1: 4 lần n=500 trong 68.0 ms
Cách 2: 4 lần n=500 trong 0.2 ms
Cách 3: 4 lần n=500 trong 0.2 ms
Cách 4: 4 lần n=500 trong 0.3 ms
Cách 5: 4 lần n=500 trong 0.1 ms
Cách 5 với n=10^6 (ngoài ràng buộc, minh họa follow-up): 929970 trong 33 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [21]:
sols = [Solution(), SolutionDeque(), SolutionRecursive(), SolutionIterative()]
edge = examples + [
    ((1, 1), 1),
    ((2, 1), 2),          # k = 1: lần lượt loại 1, 2, ... người cuối thắng
    ((2, 2), 1),
    ((7, 3), 4),          # Josephus cổ điển
    ((500, 1), 500),
]
for (n, k), want in edge:
    assert winner_mark(n, k) == want == winner_list(n, k)
    for s in sols:
        assert s.findTheWinner(n, k) == want, (n, k, s)
    print("OK", (n, k), "->", want)

OK (5, 2) -> 3
OK (6, 5) -> 1
OK (1, 1) -> 1
OK (2, 1) -> 2
OK (2, 2) -> 1
OK (7, 3) -> 4
OK (500, 1) -> 500


(Các giá trị ngoài đề đã được kiểm bằng hai mô phỏng `winner_mark`, `winner_list` trong cùng `assert`.)

### 12.2 Kiểm tra **toàn bộ** miền nhỏ và ngẫu nhiên trong ràng buộc

In [22]:
for n in range(1, 61):
    for k in range(1, n + 1):
        want = winner_list(n, k)
        assert winner_mark(n, k) == want
        for s in sols:
            assert s.findTheWinner(n, k) == want, (n, k)
rng = random.Random(1823)
for _ in range(200):
    n = rng.randint(1, 500); k = rng.randint(1, n)
    want = winner_list(n, k)
    assert all(s.findTheWinner(n, k) == want for s in sols)
print("mọi (n, k) với n ≤ 60 + 200 cặp ngẫu nhiên n ≤ 500: khớp")

mọi (n, k) với n ≤ 60 + 200 cặp ngẫu nhiên n ≤ 500: khớp


### 12.3 Kiểm tra thứ tự bị loại (Cách 2 vs Cách 6 Fenwick vs mô phỏng đánh dấu)

Ví dụ 2 của đề nêu rõ thứ tự rời vòng `5, 4, 6, 2, 3` — dùng làm test, rồi đối chiếu ngẫu nhiên.

In [23]:
def order_mark(n, k):
    alive = [True] * (n + 1); cur = 1; out = []
    for _ in range(n - 1):
        cnt = 0
        while True:
            if alive[cur]:
                cnt += 1
                if cnt == k: break
            cur = cur % n + 1
        alive[cur] = False; out.append(cur)
        cur = cur % n + 1
        while not alive[cur]: cur = cur % n + 1
    return out

def order_list(n, k):
    people = list(range(1, n + 1)); idx = 0; out = []
    while len(people) > 1:
        idx = (idx + k - 1) % len(people); out.append(people.pop(idx)); idx %= len(people)
    return out

assert order_list(6, 5) == [5, 4, 6, 2, 3] and order_list(5, 2) == [2, 4, 1, 5]
rng = random.Random(77)
for _ in range(300):
    n = rng.randint(1, 120); k = rng.randint(1, n)
    o = order_list(n, k)
    assert o == order_mark(n, k) == elimination_fenwick(n, k)[0]
print("thứ tự bị loại: 3 cách khớp trên 300 cặp (n, k) ngẫu nhiên")

thứ tự bị loại: 3 cách khớp trên 300 cặp (n, k) ngẫu nhiên


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đi `k` bước thay vì `k − 1`

In [24]:
def wrong_k_steps(n, k):
    people = list(range(1, n + 1)); idx = 0
    while len(people) > 1:
        idx = (idx + k) % len(people)          # SAI: đếm k người tính cả người bắt đầu = k-1 bước
        people.pop(idx); idx %= len(people)
    return people[0]

print("sai:", wrong_k_steps(5, 2), "| đúng:", Solution().findTheWinner(5, 2))

sai: 4 | đúng: 3


**Vì sao sai:** "đếm 2 người **kể cả** người bắt đầu" là người bắt đầu (bước 0) và người kế (bước 1) ⇒ đi `k − 1` bước. Lệch 1 làm loại sai người ngay từ lượt đầu (loại 3 thay vì 2).

### Lỗi 2 — Sau khi `pop`, lại tăng chỉ số thêm 1

In [25]:
def wrong_advance(n, k):
    people = list(range(1, n + 1)); idx = 0
    while len(people) > 1:
        idx = (idx + k - 1) % len(people)
        people.pop(idx)
        idx = (idx + 1) % len(people)          # SAI: người kế tiếp đã dồn vào chỉ số idx
    return people[0]

for n, k in [(5, 2), (6, 5), (7, 3), (4, 2)]:
    print((n, k), "sai:", wrong_advance(n, k), "| đúng:", Solution().findTheWinner(n, k))

(5, 2) sai: 3 | đúng: 3
(6, 5) sai: 3 | đúng: 1
(7, 3) sai: 1 | đúng: 4
(4, 2) sai: 4 | đúng: 1


**Vì sao sai:** sau `pop(idx)`, các phần tử phía sau dồn lên — người "ngay sau người vừa rời" đã ở chỉ số `idx`. Tăng thêm 1 là bỏ qua người đó. Để ý `(5, 2)` **tình cờ** vẫn ra đúng 3 (ví dụ 1 của đề không bắt được lỗi!), nhưng `(6, 5)` (ví dụ 2) ra 3 thay vì 1. **Sửa:** `idx %= len(people)`.

### Lỗi 3 — Quên đổi 0-based sang 1-based

In [26]:
def wrong_index_base(n, k):
    res = 0
    for m in range(2, n + 1):
        res = (res + k) % m
    return res                                  # SAI: thiếu + 1

for (n, k), want in examples:
    print((n, k), "sai:", wrong_index_base(n, k), "| đúng:", want)

(5, 2) sai: 2 | đúng: 3
(6, 5) sai: 0 | đúng: 1


**Vì sao sai:** công thức Josephus làm việc với vị trí `0..n−1`; đề đánh số `1..n`. **Sửa:** trả `res + 1`. (Một biến thể sai khác: dùng `(res + k) % m` nhưng khởi tạo `res = 1` — trộn lẫn hai hệ đánh số.)

### Lỗi 4 — Đệ quy khi `n` lớn (follow-up)

In [27]:
def josephus_rec(m, k):
    return 0 if m == 1 else (josephus_rec(m - 1, k) + k) % m

try:
    josephus_rec(5000, 3)
    print("không lỗi")
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "| vòng lặp:", Solution().findTheWinner(5000, 3))

RecursionError: maximum recursion depth exceeded | vòng lặp: 2864


**Vì sao sai:** với ràng buộc `n ≤ 500` đệ quy vẫn chạy, nhưng follow-up hướng tới `n` lớn — đệ quy sâu `n` tầng vượt giới hạn 1000 của Python. **Sửa:** vòng lặp (Cách 5), cũng đạt bộ nhớ `O(1)`.

### Lỗi 5 — Lấy modulo theo `n` cố định thay vì số người hiện tại `m`

In [28]:
def wrong_mod_n(n, k):
    res = 0
    for m in range(2, n + 1):
        res = (res + k) % n                    # SAI: vòng m người phải lấy % m
    return res % n + 1

for n, k in [(5, 2), (6, 5), (7, 3)]:
    print((n, k), "sai:", wrong_mod_n(n, k), "| đúng:", Solution().findTheWinner(n, k))

(5, 2) sai: 4 | đúng: 3
(6, 5) sai: 2 | đúng: 1
(7, 3) sai: 5 | đúng: 4


**Vì sao sai:** ở bước `m`, ta đang "dịch" vị trí trên vòng **`m` người**; vị trí hợp lệ là `0..m−1`, nên phải `% m`. Dùng `% n` cho vị trí có thể vượt `m − 1` — vô nghĩa trên vòng nhỏ. **Sửa:** `res = (res + k) % m`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Ngồi vòng tròn, đếm `k`, loại người, tiếp tục"** ⇒ Josephus: `res = (res + k) % m` cho `m = 2..n`, đáp án `res + 1`.
- Mô phỏng vòng tròn bằng list: `idx = (idx + k − 1) % len`, sau `pop` **không** tăng `idx`.
- `deque.rotate(-(k−1))` + `popleft()` là cách mô phỏng ít lỗi nhất.
- Công thức truy hồi "giải bài `n` từ bài `n − 1` rồi **dịch modulo**" là kỹ thuật dùng lại được.
- Luôn kiểm chéo công thức với mô phỏng trên mọi `(n, k)` nhỏ.

## 15. Biến thể của bài

1. **`k = 2` có công thức đóng:** viết `n = 2ᵃ + L` (`0 ≤ L < 2ᵃ`) thì người thắng là `2L + 1` (Bài tập 4). *Trực giác:* khi số người là lũy thừa của 2, mỗi vòng quanh loại đúng một nửa và người bắt đầu (số 1) luôn sống sót ⇒ thắng. Với `n = 2ᵃ + L`, sau `L` lượt loại (người 2, 4, …, 2L) còn đúng `2ᵃ` người và người bắt đầu lượt tiếp theo là `2L + 1` ⇒ người đó thắng.
2. **Bắt đầu từ người `s`** thay vì người 1: dịch kết quả `((J(n) + s − 1) mod n) + 1` (Bài tập 5).
3. **Thứ tự bị loại** (không chỉ người thắng): mô phỏng; với `n` lớn dùng cây Fenwick/segment tree để tìm "người còn sống thứ `t`" trong `O(log n)`.
4. **`n` cực lớn, `k` nhỏ:** có thuật toán `O(k log n)` nhảy nhiều lượt một lúc.
5. LeetCode 390 *Elimination Game* (loại xen kẽ hai chiều) — cùng tinh thần "đệ quy trên bài nhỏ hơn rồi ánh xạ lại".

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đánh dấu | đếm từng người, bỏ qua người đã rời | `O(n² + nk)` | `O(n)` | đúng nghĩa đen, kiểm chứng |
| 2. List + pop | chỉ số modulo | `O(n²)` | `O(n)` | cần thứ tự bị loại |
| 3. Deque | rotate + popleft | `O(nk)` | `O(n)` | code ngắn, ít lỗi chỉ số |
| 4. Đệ quy Josephus | `J(m) = (J(m−1)+k) mod m` | `O(n)` | `O(n)` | hiểu công thức |
| 5. Vòng lặp Josephus | như 4, không stack | `O(n)` | `O(1)` | **nộp bài**, follow-up |
| 6. Fenwick | "người còn sống thứ t" bằng BIT | `O(n log n)` | `O(n)` | cần thứ tự bị loại với `n` lớn |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `findTheWinner(1, 1)`, `findTheWinner(7, 3)`, `findTheWinner(4, 4)`?

**Bài 2 (tính tay):** Lập bảng `J(m)` cho `n = 5, k = 2` bằng công thức và so với mô phỏng ở mục 4.

**Bài 3 (điền TODO):** Viết `elimination_order(n, k)` trả về list thứ tự những người bị loại (độ dài `n − 1`).

**Bài 4 (tự viết lại):** Viết `winner_k2(n)` dùng công thức đóng cho `k = 2` (không vòng lặp theo `n`), kiểm với `Solution`.

**Bài 5 (mini challenge):** Viết `winner_from(n, k, s)` — người thắng nếu trò chơi bắt đầu đếm từ người `s` (thay vì 1). Kiểm bằng mô phỏng.

In [29]:
def elimination_order(n, k):
    # TODO
    pass

def winner_k2(n):
    # TODO: n = 2^a + L -> 2L + 1
    pass

def winner_from(n, k, s):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `n = 1` thì chính người đó | `(7,3)` là Josephus cổ điển | `(4,4)`: rời 4, 1, 3 ⇒ 2 |
| 2 | `J(1) = 0` | `J(2) = 0`, `J(3) = 2` | `J(4) = 0`, `J(5) = 2` ⇒ 3 |
| 3 | Dùng Cách 2 | Ghi lại phần tử bị `pop` | Độ dài kết quả `n − 1` |
| 4 | `a = n.bit_length() − 1` | `L = n − 2ᵃ` | Trả `2L + 1` |
| 5 | Trò chơi từ người `s` là trò chơi gốc "xoay" `s − 1` vị trí | `(J(n) + s − 1) mod n` | `+ 1` để về 1-based |

## 19. Đáp án tham khảo

In [30]:
r1 = [Solution().findTheWinner(*a) for a in [(1, 1), (7, 3), (4, 4)]]
print("Bài 1:", r1); assert r1 == [1, 4, 2]
print("Bài 2:"); assert trace_josephus(5, 2) == 3

Bài 1: [1, 4, 2]
Bài 2:
m=1: J=0
m=2: (0 + 2) % 2 = 0
m=3: (0 + 2) % 3 = 2
m=4: (2 + 2) % 4 = 0
m=5: (0 + 2) % 5 = 2
người thắng (1-based): 3


In [31]:
def elimination_order(n, k):
    people = list(range(1, n + 1)); idx = 0; out = []
    while len(people) > 1:
        idx = (idx + k - 1) % len(people)
        out.append(people.pop(idx))
        idx %= len(people)
    return out

def winner_k2(n):
    a = n.bit_length() - 1
    L = n - (1 << a)
    return 2 * L + 1

assert elimination_order(5, 2) == [2, 4, 1, 5]
assert elimination_order(6, 5) == [5, 4, 6, 2, 3]
for n in range(1, 2001):
    assert winner_k2(n) == Solution().findTheWinner(n, 2)
print("Bài 3, 4: OK", elimination_order(7, 3), [winner_k2(n) for n in range(1, 11)])

Bài 3, 4: OK [3, 6, 2, 7, 5, 1] [1, 1, 3, 1, 3, 5, 7, 1, 3, 5]


In [32]:
def winner_from(n, k, s):
    res = 0
    for m in range(2, n + 1):
        res = (res + k) % m
    return (res + s - 1) % n + 1

def winner_from_sim(n, k, s):
    people = list(range(1, n + 1)); idx = s - 1
    while len(people) > 1:
        idx = (idx + k - 1) % len(people)
        people.pop(idx); idx %= len(people)
    return people[0]

for n in range(1, 40):
    for k in range(1, n + 1):
        for s in range(1, n + 1):
            assert winner_from(n, k, s) == winner_from_sim(n, k, s)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", winner_from(5, 2, 3))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK 5


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
res = 0                     # J(1) = 0, vị trí 0-based
for m in range(2, n + 1):
    res = (res + k) % m     # J(m) = (J(m-1) + k) mod m
return res + 1              # về đánh số 1..n

# mô phỏng (khi cần thứ tự loại):
idx = (idx + k - 1) % len(people); people.pop(idx); idx %= len(people)
```

**Checklist:**
- [ ] Đếm `k` người **kể cả** người bắt đầu ⇒ `k − 1` bước?
- [ ] Sau `pop(idx)` **không** tăng `idx` (chỉ `idx %= len`)?
- [ ] Công thức dùng `% m` với `m` là **số người hiện tại**, không phải `% n`?
- [ ] Trả `res + 1` (1-based)?
- [ ] `n = 1` trả 1; dùng vòng lặp thay đệ quy cho `n` lớn.